**Goal**

1. একটা “today capture” এর জন্য একটাই JSON/CSV report হবে:

2. identity status (CoreEngine similarity-based)

3. skin scores (stable v2)

4. emotion + age

5. alerts (spike rules)

6. metadata (image path, time)

In [12]:
import os, re, json, datetime
import numpy as np
import pandas as pd

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
# 1) CoreEngine output optional (যদি similarity alert JSON/CSV বানাও)
# (CoreEngine এখন print করে; later export add করবো)

# 2) Skin stable v2 CSV (তুমি save করেছো)
SKIN_CSV = "/content/drive/MyDrive/Mirror Vision-KARIGOR/MirrorVision_ROI_Output_IF/skin_scores_stable_v2.csv"

# 3) Emotion+Age CSV
EMO_AGE_CSV = "/content/drive/MyDrive/Mirror Vision-KARIGOR/Dataset_Mirror_SPLIT/MirrorVision_EmotionAge_Output/emotion_age_v2_clean.csv"
#EMO_AGE_CSV = "/content/drive/MyDrive/MirrorVision_EmotionAge_Output/emotion_age_v2_clean.csv"

# 4) Output report location
OUT_DIR = "/content/drive/MyDrive/Mirror Vision-KARIGOR/Dataset_Mirror_SPLIT/Mirror Vision-KARIGOR_daily_report.csv"
os.makedirs(OUT_DIR, exist_ok=True)

print("Skin CSV exists:", os.path.exists(SKIN_CSV))
print("EmotionAge CSV exists:", os.path.exists(EMO_AGE_CSV))
print("Out dir:", OUT_DIR)

Skin CSV exists: True
EmotionAge CSV exists: True
Out dir: /content/drive/MyDrive/Mirror Vision-KARIGOR/Dataset_Mirror_SPLIT/Mirror Vision-KARIGOR_daily_report.csv


In [25]:
#Load CSV
skin = pd.read_csv(SKIN_CSV)
emo = pd.read_csv(EMO_AGE_CSV)

print("skin shape:", skin.shape)
print("emo shape:", emo.shape)

skin.head(), emo.head()

skin shape: (44, 5)
emo shape: (44, 7)


(              image_id  dark_circle_score_stable  wrinkle_score_stable  \
 0  IMG-20260112-WA0000                  0.000000             65.530957   
 1  IMG-20260112-WA0001                  0.000000             89.603506   
 2  IMG-20260112-WA0002                  0.000000            100.000000   
 3  IMG-20260112-WA0003                  7.395097             81.528900   
 4  IMG-20260112-WA0004                  0.000000             49.070630   
 
    acne_count_v2  acne_severity_v2_0_100  
 0              0                0.000000  
 1              0                0.000000  
 2              0                0.000000  
 3              1                8.658009  
 4              1                8.658009  ,
                                                 path                    image  \
 0  /content/drive/MyDrive/Mirror Vision-KARIGOR/D...  IMG-20260112-WA0000.jpg   
 1  /content/drive/MyDrive/Mirror Vision-KARIGOR/D...  IMG-20260112-WA0001.jpg   
 2  /content/drive/MyDrive/Mirror Vis

Date parser

In [26]:
def parse_date_from_id(image_id: str):
    m = re.search(r'(20\d{6})', str(image_id))
    if m:
        return pd.to_datetime(m.group(1), format="%Y%m%d", errors="coerce")
    return pd.NaT

# skin image_id (folder name) থেকে date
skin["date"] = skin["image_id"].apply(parse_date_from_id)

# emo image filename থেকে date
emo["image_id"] = emo["image"].astype(str).str.replace(".jpg","", regex=False)
emo["date"] = emo["image_id"].apply(parse_date_from_id)

print("skin date range:", skin["date"].min(), "->", skin["date"].max())
print("emo  date range:", emo["date"].min(), "->", emo["date"].max())

skin date range: 2026-01-12 00:00:00 -> 2026-01-16 00:00:00
emo  date range: 2026-01-12 00:00:00 -> 2026-01-16 00:00:00


Daily aggregation (Skin + EmotionAge)

একই দিনে multiple photo থাকলে daily mean নেব।

In [27]:
# Skin daily
skin_daily = skin.dropna(subset=["date"]).groupby("date").agg(
    dark_circle=("dark_circle_score_stable","mean"),
    wrinkle=("wrinkle_score_stable","mean"),
    acne=("acne_severity_v2_0_100","mean")
).reset_index()

# Emotion/Age daily
emo_daily = emo.dropna(subset=["date"]).groupby("date").agg(
    emotion=("emotion", lambda x: x.value_counts().index[0] if len(x) else None),
    emotion_conf=("emotion_conf", "mean"),
    age_pred=("age_pred", "mean"),
    age_label=("age_label", lambda x: x.value_counts().index[0] if len(x) else None),
    face_found=("face_found", "mean")
).reset_index()

skin_daily, emo_daily

(        date  dark_circle    wrinkle       acne
 0 2026-01-12     1.091235  84.148857  11.392117
 1 2026-01-13     0.165409  84.554941  43.819144
 2 2026-01-14     0.763017  71.191401  36.839827
 3 2026-01-16     0.969409  54.449883  42.640693,
         date  emotion  emotion_conf   age_pred      age_label  face_found
 0 2026-01-12  neutral      0.775778  35.263158  Looks Younger         1.0
 1 2026-01-13      sad      0.748512  35.000000  Looks Younger         1.0
 2 2026-01-14  neutral      0.766290  33.300000  Looks Younger         1.0
 3 2026-01-16  neutral      0.868927  34.500000  Looks Younger         1.0)

Merge daily table

In [28]:
daily = pd.merge(skin_daily, emo_daily, on="date", how="outer").sort_values("date")
daily

,date,dark_circle,wrinkle,acne,emotion,emotion_conf,age_pred,age_label,face_found
0,2026-01-12,1.091235,84.148857,11.392117,neutral,0.775778,35.263158,Looks Younger,1.0
1,2026-01-13,0.165409,84.554941,43.819144,sad,0.748512,35.000000,Looks Younger,1.0
2,2026-01-14,0.763017,71.191401,36.839827,neutral,0.766290,33.300000,Looks Younger,1.0
3,2026-01-16,0.969409,54.449883,42.640693,neutral,0.868927,34.500000,Looks Younger,1.0


Spike alerts (MVP)

In [29]:
def add_spike(df, col, spike_pct=30):
    prev = df[col].shift(1)
    change = (df[col] - prev) / (prev.replace(0, np.nan))
    return (change > (spike_pct/100.0)).fillna(False)

daily = daily.sort_values("date").reset_index(drop=True)

daily["dark_circle_spike"] = add_spike(daily, "dark_circle", spike_pct=30)
daily["wrinkle_spike"]     = add_spike(daily, "wrinkle", spike_pct=20)
daily["acne_spike"]        = add_spike(daily, "acne", spike_pct=30)

daily

,date,dark_circle,wrinkle,acne,emotion,emotion_conf,age_pred,age_label,face_found,dark_circle_spike,wrinkle_spike,acne_spike
0,2026-01-12,1.091235,84.148857,11.392117,neutral,0.775778,35.263158,Looks Younger,1.0,False,False,False
1,2026-01-13,0.165409,84.554941,43.819144,sad,0.748512,35.000000,Looks Younger,1.0,False,False,True
2,2026-01-14,0.763017,71.191401,36.839827,neutral,0.766290,33.300000,Looks Younger,1.0,True,False,False
3,2026-01-16,0.969409,54.449883,42.640693,neutral,0.868927,34.500000,Looks Younger,1.0,False,False,False


Make JSON daily reports

In [30]:
def to_float(x):
    try:
        if pd.isna(x): return None
        return float(x)
    except:
        return None

def to_str(x):
    if pd.isna(x): return None
    return str(x)

reports = []

for _, r in daily.iterrows():
    date_str = r["date"].strftime("%Y-%m-%d") if not pd.isna(r["date"]) else None

    report = {
        "date": date_str,
        "identity_status": "Stable",  # CoreEngine integrate করলে এখানে update হবে
        "skin": {
            "dark_circle": to_float(r.get("dark_circle")),
            "wrinkle": to_float(r.get("wrinkle")),
            "acne_severity": to_float(r.get("acne")),
        },
        "emotion": {
            "label": to_str(r.get("emotion")),
            "confidence": to_float(r.get("emotion_conf")),
        },
        "age": {
            "predicted": to_float(r.get("age_pred")),
            "appearance": to_str(r.get("age_label")),
        },
        "alerts": {
            "dark_circle_spike": bool(r.get("dark_circle_spike", False)),
            "wrinkle_spike": bool(r.get("wrinkle_spike", False)),
            "acne_spike": bool(r.get("acne_spike", False)),
        },
        "meta": {
            "face_found_rate": to_float(r.get("face_found")),
            "generated_at": datetime.datetime.now().isoformat(timespec="seconds")
        }
    }

    reports.append(report)

# save all in one JSON file
all_json_path = os.path.join(OUT_DIR, "daily_reports_v1.json")
with open(all_json_path, "w", encoding="utf-8") as f:
    json.dump(reports, f, ensure_ascii=False, indent=2)

print("Saved:", all_json_path)
print("Total daily reports:", len(reports))

# print first report
reports[0] if reports else None

Saved: /content/drive/MyDrive/Mirror Vision-KARIGOR/Dataset_Mirror_SPLIT/Mirror Vision-KARIGOR_daily_report.csv/daily_reports_v1.json
Total daily reports: 4


{'date': '2026-01-12',
 'identity_status': 'Stable',
 'skin': {'dark_circle': 1.0912350159125983,
  'wrinkle': 84.14885728231256,
  'acne_severity': 11.392116655274545},
 'emotion': {'label': 'neutral', 'confidence': 0.7757779332211143},
 'age': {'predicted': 35.26315789473684, 'appearance': 'Looks Younger'},
 'alerts': {'dark_circle_spike': False,
  'wrinkle_spike': False,
  'acne_spike': False},
 'meta': {'face_found_rate': 1.0, 'generated_at': '2026-02-28T11:01:31'}}

Save a single “latest day” JSON (app friendly)

In [31]:
if reports:
    latest = reports[-1]
    latest_path = os.path.join(OUT_DIR, "latest_report.json")
    with open(latest_path, "w", encoding="utf-8") as f:
        json.dump(latest, f, ensure_ascii=False, indent=2)
    print("Saved latest:", latest_path)
    latest

Saved latest: /content/drive/MyDrive/Mirror Vision-KARIGOR/Dataset_Mirror_SPLIT/Mirror Vision-KARIGOR_daily_report.csv/latest_report.json


Export merged daily table CSV (for boss + dashboard)

In [32]:
out_csv = os.path.join(OUT_DIR, "daily_reports_table_v1.csv")
daily.to_csv(out_csv, index=False)
print("Saved:", out_csv)
daily

Saved: /content/drive/MyDrive/Mirror Vision-KARIGOR/Dataset_Mirror_SPLIT/Mirror Vision-KARIGOR_daily_report.csv/daily_reports_table_v1.csv


,date,dark_circle,wrinkle,acne,emotion,emotion_conf,age_pred,age_label,face_found,dark_circle_spike,wrinkle_spike,acne_spike
0,2026-01-12,1.091235,84.148857,11.392117,neutral,0.775778,35.263158,Looks Younger,1.0,False,False,False
1,2026-01-13,0.165409,84.554941,43.819144,sad,0.748512,35.000000,Looks Younger,1.0,False,False,True
2,2026-01-14,0.763017,71.191401,36.839827,neutral,0.766290,33.300000,Looks Younger,1.0,True,False,False
3,2026-01-16,0.969409,54.449883,42.640693,neutral,0.868927,34.500000,Looks Younger,1.0,False,False,False
